# Import other information
Add other information to the database that will be useful for queries.

## Prerequisites
This notebook assumes that you have already run the `import_socat` notebook.

In [ ]:
import sqlite3
from osgeo import ogr
import pandas as pd
from sqlalchemy.types import NVARCHAR

DB_FILE = 'socat_kpi.sqlite'

conn = sqlite3.connect(DB_FILE)
conn.enable_load_extension(True)
conn.load_extension("mod_spatialite")


## ICES Countries
The first two characters of the EXPO Code are the platform's ICES country code. We add the complete list of ICES country codes for easier presentation.

In [ ]:
conn.execute("DROP TABLE IF EXISTS countries")
conn.commit()

conn.execute("""CREATE TABLE countries(
code VARCHAR(5),
name VARCHAR(100)
)""")

countries_df = pd.read_csv('ICES_Countries.tsv', sep='\t', index_col=False)
countries_df.to_sql('countries', conn, if_exists='append', index=False)

conn.execute('CREATE INDEX countries_code ON countries(code)')
conn.commit()


# PI Countries
The ICES country codes assigned in the EXPO Codes are often not useful to us. We are interested in the country that was responsible for the observations, which is frequently different to the ship's country (most commonly due to ships sailing under flags of convenience).

In [ ]:
conn.execute("DROP TABLE IF EXISTS pi_countries")
conn.commit()

conn.execute("""CREATE TABLE pi_countries(
expocode VARCHAR(20),
platform_name VARCHAR(100),
platform_type VARCHAR(20),
organization VARCHAR(100),
investigators VARCHAR(200),
country VARCHAR(100)
)""")

pi_df = pd.read_csv('pi_countries.csv', index_col=False)
pi_df.to_sql('pi_countries', conn, if_exists='append', index=False)

conn.execute('CREATE INDEX pi_expocode ON pi_countries(expocode)')
conn.commit()


## Close Down

In [ ]:
conn.close()